# 02 - Data Quality Assessment & Preparation

**Objective:** verify the raw data against its documentation, document anomalies, and produce validated analytical tables.
**Grain:** both raw tables, checked separately and then cross-checked against each other.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", message="findfont")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.visualization.style import apply_matplotlib_style
apply_matplotlib_style()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 40)

## Automated checks
All checks live in `src/data/validate.py`, so the notebook and the pipeline run identical logic.

In [2]:
from src.data.validate import run_all_checks, missing_hours_table
from src.data.ingest import load_raw
checks = pd.DataFrame([c.__dict__ for c in run_all_checks()])
checks["status"] = checks.apply(lambda r: "PASS" if r.passed else ("FAIL" if r.blocking else "WARN"), axis=1)
display(checks[["table", "check", "status", "detail"]])
print("Blocking failures:", int(((~checks.passed) & checks.blocking).sum()))

,table,check,status,detail
0,day,No missing values,PASS,0 null cells
1,day,No duplicate rows,PASS,0 duplicate rows
2,day,Unique key ['dteday'],PASS,0 duplicate keys
3,day,cnt == casual + registered,PASS,0 mismatching rows
4,day,"season within [1, 4]",PASS,0 out of range
5,day,"yr within [0, 1]",PASS,0 out of range
6,day,"mnth within [1, 12]",PASS,0 out of range
7,day,"holiday within [0, 1]",PASS,0 out of range
8,day,"weekday within [0, 6]",PASS,0 out of range
9,day,"workingday within [0, 1]",PASS,0 out of range


Blocking failures: 0


## Finding 1 - Missing hours
`hour.csv` should hold 731 x 24 = 17,544 rows but holds 17,379.

In [3]:
hour = load_raw("hour"); day = load_raw("day")
gaps = missing_hours_table(hour)
print(f"{len(gaps)} missing hour slots on {gaps.dteday.nunique()} days")
display(gaps.groupby("dteday").size().sort_values(ascending=False).head(8).rename("hours missing"))
display(gaps["hr"].value_counts().sort_index().rename("missing slots by hour").to_frame().T)

165 missing hour slots on 76 days


dteday
2012-10-29    23
2011-01-27    16
2012-10-30    13
2011-01-18    12
2011-01-26     8
2011-08-28     7
2011-08-27     6
2011-02-22     6
Name: hours missing, dtype: int64

hr,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
missing slots by hour,5,7,16,34,34,14,6,4,4,4,4,4,3,2,2,2,1,1,3,3,3,3,3,3


59% of missing slots fall between 02:00 and 05:00, when demand is lowest. The largest gaps are 2012-10-29 (23 of 24 hours missing; Hurricane Sandy), 2011-01-27 (16 hours), 2012-10-30 (13) and 2011-01-18 (12). Summed hourly counts match `day.csv` on every day, so the source aggregation has **no rentals in these hours**. They are most likely zero-rental or outage hours.

**Decision:** do not impute. Hourly means are computed over observed hours only. This slightly overstates the mean for the quietest night hours, and the notebooks state that in their limitations.

## Finding 2 - Season labels contradict the dates

In [4]:
pd.crosstab(day["season"], day["mnth"])

mnth,1,2,3,4,5,6,7,8,9,10,11,12
season,,,,,,,,,,,,
1,62,57,40,0,0,0,0,0,0,0,0,22
2,0,0,22,60,62,40,0,0,0,0,0,0
3,0,0,0,0,0,20,62,62,44,0,0,0
4,0,0,0,0,0,0,0,0,16,62,60,40


The Readme says `1: springer, 2: summer, 3: fall, 4: winter`. However, code 1 contains **all of January and February** and the dates from about 21 December to 20 March, which is astronomical winter. Code 3 contains July and August.

**Decision:** label the codes by their dates (1 Winter, 2 Spring, 3 Summer, 4 Fall). The original `season` column is kept. Keeping the Readme labels would make every seasonal finding wrong. For example, "Spring" would turn out to be the lowest-demand season.

## Finding 3 - Implausible humidity and a sparse weather category

In [5]:
display(hour.loc[hour.hum == 0, ["dteday", "hr", "hum", "weathersit", "cnt"]].head())
display(pd.DataFrame({"daily days": day.weathersit.value_counts().sort_index(),
                      "hourly hours": hour.weathersit.value_counts().sort_index()}))

,dteday,hr,hum,weathersit,cnt
1551,2011-03-10,0,0.00,3,3
1552,2011-03-10,1,0.00,3,2
1553,2011-03-10,2,0.00,3,1
1554,2011-03-10,5,0.00,3,3
1555,2011-03-10,6,0.00,3,12


,daily days,hourly hours
weathersit,,
1,463.00,11413
2,247.00,4544
3,21.00,1419
4,NaN,3


- Humidity is exactly 0% for every hour of 2011-03-10, a rainy day. This is a sensor/recording error, so `hum_pct` is set to missing. The rental counts are kept.
- Weather category 4 occurs in only 3 hours and never at daily grain. It is merged with category 3 into **"Rain / snow"** for group comparisons.

## Preparation
Run `src/data/clean.py`. It adds labelled categoricals, de-normalized weather columns, `day_type` and hour buckets, preserves every original column, and asserts that row counts are unchanged.

In [6]:
from src.data.clean import run_pipeline, load_processed
counts = run_pipeline(); print("Rows written:", counts)
d, h = load_processed("day"), load_processed("hour")
assert len(d) == len(day) and len(h) == len(hour), "row count changed"
assert (d.cnt == d.casual + d.registered).all() and (h.cnt == h.casual + h.registered).all()
display(d[["temp", "temp_c", "atemp_c", "hum_pct", "windspeed_kmh"]].describe().T)
display(d.day_type.value_counts().rename("days"))
print("New columns:", sorted(set(d.columns) - set(day.columns)))

Rows written: {'day': 731, 'hour': 17379}


,count,mean,std,min,25%,50%,75%,max
temp,731.00,0.50,0.18,0.06,0.34,0.50,0.66,0.86
temp_c,731.00,20.31,7.51,2.42,13.82,20.43,26.87,35.33
atemp_c,731.00,23.72,8.15,3.95,16.89,24.34,30.43,42.04
hum_pct,730.00,62.88,14.06,18.79,52.05,62.71,73.03,97.25
windspeed_kmh,731.00,12.76,5.19,1.50,9.04,12.13,15.63,34.00


day_type
Working day    500
Weekend        210
Holiday         21
Name: days, dtype: int64

New columns: ['atemp_c', 'casual_share', 'day_type', 'hum_pct', 'month', 'month_label', 'season_label', 'temp_c', 'weather_group', 'weather_label', 'weekday_label', 'windspeed_kmh', 'year', 'year_month']


## Conclusions & limitations
- The data is clean structurally: no nulls, no duplicates, consistent calendar fields, and counts that reconcile across grains.
- Three documented decisions: missing hours are not imputed, season labels are corrected from the dates, and zero humidity is set to missing (plus categories 3 and 4 merged).
- The full check log is in `reports/findings/data_quality.md`, and the validated tables are in `data/processed/`.